<a href="https://colab.research.google.com/github/jmc929/Modelos1/blob/main/03_Escalado%2BOneHot_SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dependencias

In [ ]:
!pip install -q kaggle

import os, json, time
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

from datetime import datetime
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

RND = 42
np.random.seed(RND)

# Carga de datos desde Kaggle

In [ ]:
!wget --no-cache -O init.py -q https://raw.githubusercontent.com/rramosp/ai4eng.v1/main/content/init.py
import init; init.init(force_download=False); init.get_weblink()

os.environ['KAGGLE_CONFIG_DIR'] = '/content/'
data = {"username":"cmosquera15","key":"ad6af1b3307521c527205d333e396e07"}
with open('kaggle.json','w') as f: json.dump(data, f)
!chmod 600 kaggle.json
!kaggle competitions download -c udea-ai-4-eng-20252-pruebas-saber-pro-colombia -q
!unzip -q '*.zip'

df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
print("train shape:", df.shape, " test shape:", test_df.shape)

404 Client Error: Not Found for url: https://www.kaggle.com/api/v1/competitions/data/download-all/udea-ai-4-eng-20252-pruebas-saber-pro-colombia
replace submission_example.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
train shape: (692500, 21)  test shape: (296786, 20)


# Inspección ligera

In [ ]:
print("Valores faltantes por columna (train):")
print(df.isna().sum())
print("\nTipos:")
print(df.dtypes)
print("\nDistribución objetivo:")
print(df["RENDIMIENTO_GLOBAL"].value_counts())

Valores faltantes por columna (train):
ID                                 0
PERIODO_ACADEMICO                  0
E_PRGM_ACADEMICO                   0
E_PRGM_DEPARTAMENTO                0
E_VALORMATRICULAUNIVERSIDAD     6287
E_HORASSEMANATRABAJA           30857
F_ESTRATOVIVIENDA              32137
F_TIENEINTERNET                26629
F_EDUCACIONPADRE               23178
F_TIENELAVADORA                39773
F_TIENEAUTOMOVIL               43623
E_PRIVADO_LIBERTAD                 0
E_PAGOMATRICULAPROPIO           6498
F_TIENECOMPUTADOR              38103
F_TIENEINTERNET.1              26629
F_EDUCACIONMADRE               23664
RENDIMIENTO_GLOBAL                 0
INDICADOR_1                        0
INDICADOR_2                        0
INDICADOR_3                        0
INDICADOR_4                        0
dtype: int64

Tipos:
ID                               int64
PERIODO_ACADEMICO                int64
E_PRGM_ACADEMICO                object
E_PRGM_DEPARTAMENTO             object
E_VALOR

# Imputación mínima y mapeos previos

In [ ]:
df = df.copy()
test_df = test_df.copy()

rend_map = {'bajo': 0, 'medio-bajo': 1, 'medio-alto': 2, 'alto': 3}
y_full = df['RENDIMIENTO_GLOBAL'].map(rend_map).astype(int)
df = df.drop(columns=['RENDIMIENTO_GLOBAL'])

cat_fill_noinfo = ['F_EDUCACIONMADRE', 'F_EDUCACIONPADRE', 'E_VALORMATRICULAUNIVERSIDAD',
                   'F_ESTRATOVIVIENDA', 'E_PRGM_ACADEMICO', 'E_PRGM_DEPARTAMENTO']
for c in cat_fill_noinfo:
    if c in df.columns:
        df[c] = df[c].fillna('no info')
    if c in test_df.columns:
        test_df[c] = test_df[c].fillna('no info')

cmap = {
    'Menos de 500 mil': 0.25,
    'Entre 500 mil y menos de 1 millón': 0.75,
    'Entre 1 millón y menos de 2.5 millones': 1.75,
    'Entre 2.5 millones y menos de 4 millones': 3.25,
    'Entre 4 millones y menos de 5.5 millones': 4.75,
    'Entre 5.5 millones y menos de 7 millones': 6.25,
    'Más de 7 millones': 7.75,
    'No pagó matrícula': 0.0,
    'no info': -1.0
}

def map_matricula_col(df_local):
    if 'E_VALORMATRICULAUNIVERSIDAD' in df_local.columns:
        if df_local['E_VALORMATRICULAUNIVERSIDAD'].dtype == object:
            df_local['E_VALORMATRICULANUM'] = (
                df_local['E_VALORMATRICULAUNIVERSIDAD']
                .replace({'No sabe':'no info','No Aplica':'no info'})
                .map(cmap)
            )
        else:
            df_local['E_VALORMATRICULANUM'] = pd.to_numeric(df_local['E_VALORMATRICULAUNIVERSIDAD'], errors='coerce')
    else:
        df_local['E_VALORMATRICULANUM'] = -1.0

map_matricula_col(df)
map_matricula_col(test_df)

hours_map = {
    '0': 0, '0 horas': 0, 'Menos de 10 horas': 5, 'Menos de 10 horas.': 5,
    'Entre 10 y 20 horas': 15, 'Entre 11 y 20 horas': 15,
    'Entre 21 y 30 horas': 25, 'Entre 21 y 30 horas.': 25,
    'Más de 30 horas': 40, 'no info': -1
}
def map_hours(df_local):
    if 'E_HORASSEMANATRABAJA' in df_local.columns:
        if df_local['E_HORASSEMANATRABAJA'].dtype == object:
            df_local['E_HORASSEMANATRABAJA_NUM'] = (
                df_local['E_HORASSEMANATRABAJA']
                .replace({'No sabe':'no info','No Aplica':'no info'})
                .map(lambda x: hours_map.get(x, -1))
            )
        else:
            df_local['E_HORASSEMANATRABAJA_NUM'] = pd.to_numeric(df_local['E_HORASSEMANATRABAJA'], errors='coerce').fillna(-1)
    else:
        df_local['E_HORASSEMANATRABAJA_NUM'] = -1

map_hours(df)
map_hours(test_df)

bin_cols = [c for c in ['F_TIENEINTERNET','F_TIENELAVADORA','F_TIENEAUTOMOVIL',
                        'E_PRIVADO_LIBERTAD','E_PAGOMATRICULAPROPIO','F_TIENECOMPUTADOR',
                        'F_TIENEINTERNET.1'] if c in df.columns or c in test_df.columns]

def binarize_col(series):
    return series.fillna('no info').apply(
        lambda x: 1 if str(x).strip().lower() in ['si','sí','s'] else (
                  0 if str(x).strip().lower() in ['no','n'] else -1)
    ).astype(int)

for c in bin_cols:
    if c in df.columns:
        df[c] = binarize_col(df[c])
    if c in test_df.columns:
        test_df[c] = binarize_col(test_df[c])

for ind in ['INDICADOR_1','INDICADOR_2','INDICADOR_3','INDICADOR_4']:
    if ind in df.columns:
        df[ind] = pd.to_numeric(df[ind], errors='coerce')
    if ind in test_df.columns:
        test_df[ind] = pd.to_numeric(test_df[ind], errors='coerce')

print("Imputación y mapeos iniciales terminados.")
print("train shape:", df.shape, " test shape:", test_df.shape)

Imputación y mapeos iniciales terminados.
train shape: (692500, 22)  test shape: (296786, 22)


# One-Hot

In [ ]:
df_train_idx = df.index
df_test_idx = test_df.index + len(df)

combined = pd.concat([df, test_df], axis=0, ignore_index=True)
print("Combined shape (before OHE):", combined.shape)

cat_cols_for_ohe = [c for c in ['F_EDUCACIONMADRE', 'F_EDUCACIONPADRE', 'F_ESTRATOVIVIENDA', 'E_PRGM_DEPARTAMENTO', 'E_PRGM_ACADEMICO'] if c in combined.columns]

combined_ohe = pd.get_dummies(combined, columns=cat_cols_for_ohe, drop_first=False)

columns_to_drop_after_ohe = ['ID', 'E_VALORMATRICULAUNIVERSIDAD', 'E_HORASSEMANATRABAJA']
combined_ohe = combined_ohe.drop(columns=[col for col in columns_to_drop_after_ohe if col in combined_ohe.columns])

print("Combined shape (after OHE and dropping original cols):", combined_ohe.shape)

n_train = len(df)
df_proc = combined_ohe.iloc[:n_train].copy().reset_index(drop=True)
test_proc = combined_ohe.iloc[n_train:].copy().reset_index(drop=True)

X_full = df_proc.copy()
X_test_full = test_proc.copy()

print("X_full shape:", X_full.shape, "X_test_full shape:", X_test_full.shape)

feature_columns = X_full.columns.tolist()
print("num features:", len(feature_columns))

Combined shape (before OHE): (989286, 22)
Combined shape (after OHE and dropping original cols): (989286, 1037)
X_full shape: (692500, 1037) X_test_full shape: (296786, 1037)
num features: 1037


# Imputación numérica usando únicamente medias del TRAIN y Escalado

In [ ]:
X = X_full.copy()
y = y_full.copy()

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, random_state=RND, stratify=y)
print("X_train, X_val shapes:", X_train.shape, X_val.shape)

num_cols = [c for c in ['E_VALORMATRICULANUM','E_HORASSEMANATRABAJA_NUM','INDICADOR_1','INDICADOR_2','INDICADOR_3','INDICADOR_4'] if c in X_train.columns]
print("num_cols:", num_cols)

for c in num_cols:
    median_val = X_train[c].median()
    X_train[c] = X_train[c].fillna(median_val)
    X_val[c] = X_val[c].fillna(median_val)

scaler = StandardScaler()
if num_cols:
    X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
    X_val[num_cols] = scaler.transform(X_val[num_cols])

print("Escalado aplicado a num_cols. Shapes:", X_train.shape, X_val.shape)

X_train, X_val shapes: (554000, 1037) (138500, 1037)
num_cols: ['E_VALORMATRICULANUM', 'E_HORASSEMANATRABAJA_NUM', 'INDICADOR_1', 'INDICADOR_2', 'INDICADOR_3', 'INDICADOR_4']
Escalado aplicado a num_cols. Shapes: (554000, 1037) (138500, 1037)


# GridSearch SVM

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV

# Reducir el tamaño del dataset para GridSearchCV
# Ajusta el factor de submuestreo según sea necesario para evitar el error de RAM
sample_size = 0.1 # Usar el 10% de los datos para la búsqueda
X_train_small, _, y_train_small, _ = train_test_split(X_train, y_train, test_size=1-sample_size, random_state=RND, stratify=y_train)

lsvc = LinearSVC(max_iter=10000, random_state=RND, dual=False)
param_grid_l = {'C': [0.01, 0.1, 1]}

grid_l = GridSearchCV(lsvc, param_grid_l, cv=3, scoring='accuracy', n_jobs=-1, verbose=2)
t0 = time.time()
grid_l.fit(X_train_small, y_train_small) # Usar la submuestra para la búsqueda
t1 = time.time()
print("GridSearch LinearSVC terminado en {:.2f} min".format((t1-t0)/60))

best_params = grid_l.best_params_
print("Mejores params (LinearSVC):", best_params)

# Entrenar el modelo final con el conjunto de entrenamiento completo y los mejores parámetros
chosen_model = LinearSVC(max_iter=10000, random_state=RND, dual=False, **best_params)
chosen_model.fit(X_train, y_train)

y_pred_val = chosen_model.predict(X_val)
acc_val = accuracy_score(y_val, y_pred_val)
print("Accuracy en validación (LinearSVC):", acc_val)
print(classification_report(y_val, y_pred_val))


Fitting 3 folds for each of 3 candidates, totalling 9 fits
GridSearch LinearSVC terminado en 0.36 min
Mejores params (LinearSVC): {'C': 0.01}


# Evaluación en validación

In [ ]:
y_pred_val = chosen_model.predict(X_val)
acc_val = accuracy_score(y_val, y_pred_val)
print("Accuracy en validación:", acc_val)
print(classification_report(y_val, y_pred_val))

cm = confusion_matrix(y_val, y_pred_val)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.xlabel('Predicción')
plt.ylabel('Valor real')
plt.title('Matriz de confusión — SVM')
plt.show()

# Guardar y registrar experimento

In [ ]:
joblib.dump(scaler, 'scaler_03_EscaladoOneHot.joblib')
joblib.dump(feature_columns, 'feature_columns_03_EscaladoOneHot.joblib')
joblib.dump(chosen_model, 'svm_chosen_03_EscaladoOneHot.joblib')
print("Guardados: scaler, feature_columns y modelo.")

exp_row = {
    'notebook': '03 - Escalado + OneHot y SVM',
    'model': 'SVM',
    'params': str(best_params) if 'best_params' in locals() else 'NA',
    'val_accuracy': float(acc_val) if 'acc_val' in locals() else None,
    'test_accuracy': None,
    'kaggle_score': None,
    'model_file': 'svm_chosen_03_EscaladoOneHot.joblib',
    'date': datetime.now().isoformat()
}

exp_file = 'experiments.csv'
exp_df = pd.DataFrame([exp_row])
if os.path.exists(exp_file):
    df_exp = pd.read_csv(exp_file)
    df_exp = pd.concat([df_exp, exp_df], ignore_index=True)
else:
    df_exp = exp_df
df_exp.to_csv(exp_file, index=False)
print("Experimento registrado en", exp_file)

# Preparar test_proc, alinear columnas, escalar y crear el CSV de salida

In [ ]:
test_proc = test_proc.copy()

for c in num_cols:
    if c in test_proc.columns:
        med = X_train[c].median() if c in X_train.columns else 0
        test_proc[c] = pd.to_numeric(test_proc[c], errors='coerce').fillna(med)
    else:
        test_proc[c] = X_train[c].median() if c in X_train.columns else 0

for col in feature_columns:
    if col not in test_proc.columns:
        test_proc[col] = 0

extra = set(test_proc.columns) - set(feature_columns)
if extra:
    test_proc = test_proc.drop(columns=list(extra), errors='ignore')

test_proc = test_proc[feature_columns]

if num_cols:
    test_proc[num_cols] = scaler.transform(test_proc[num_cols])

preds_num = chosen_model.predict(test_proc)

inv_map = {0:'bajo', 1:'medio-bajo', 2:'medio-alto', 3:'alto'}
preds_text = [inv_map[int(p)] for p in preds_num]

submission = pd.DataFrame({'ID': test_df['ID'].values, 'RENDIMIENTO_GLOBAL': preds_text})

out_filename = "archivo_EscaladoOneHot_SVM.csv"
submission.to_csv(out_filename, index=False)
print("Archivo creado:", out_filename, " shape:", submission.shape)

# Subir a Kaggle

In [ ]:
!kaggle competitions submit -c udea-ai-4-eng-20252-pruebas-saber-pro-colombia -f archivo_EscaladoOneHot_SVM-RBF.csv -m "03 SVM attempt"